In [1]:
!pip install torchmetrics

   ---------------------------------------- 0.0/983.4 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.4 kB ? eta -:--:--
   ---------------------------------------- 983.4/983.4 kB 3.3 MB/s  0:00:00

   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------- ------------------- 1/2 [torchmetrics]
   -------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
CHANNELS = 3

In [1]:
import os, glob, itertools, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import logging
import sys

from torchmetrics.image.fid import FrechetInceptionDistance

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True 

In [4]:
class Generator(nn.Module):
    def __init__(self, z_dim, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf, CHANNELS, 4, 2, 1, bias=False),
            nn.Tanh(),  # output w [-1, 1]
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(CHANNELS, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1)


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.constant_(m.bias, 0)

In [5]:
OUT_DIR ="."

In [6]:
# ============================
# DCGAN LATENT INTERPOLATION
# ============================

import os
import torch
import matplotlib.pyplot as plt
from torchvision.utils import save_image


def to_01(x):
    return ((x + 1) / 2).clamp(0, 1)


def load_generator_for_interpolation(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)

    z_dim = ckpt["latent_dim"]

    G = Generator(z_dim).to(device)
    G.load_state_dict(ckpt["G_state_dict"])
    G.eval()

    return G, z_dim, ckpt


@torch.no_grad()
def make_dcgan_latent_interpolation(
    checkpoint_path,
    output_name="dcgan_latent_interpolation",
    interpolation_seed=123,
):
    G, z_dim, ckpt = load_generator_for_interpolation(checkpoint_path)

    z_gen = torch.Generator(device=device)
    z_gen.manual_seed(interpolation_seed)

    # Dwa oryginalne latent vectors
    z1 = torch.randn(1, z_dim, 1, 1, device=device, generator=z_gen)
    z2 = torch.randn(1, z_dim, 1, 1, device=device, generator=z_gen)

    # 10 obrazów: z1 + 8 interpolowanych + z2
    alphas = torch.linspace(0, 1, steps=10, device=device)

    z_all = torch.cat(
        [(1 - alpha) * z1 + alpha * z2 for alpha in alphas],
        dim=0
    )

    fake = G(z_all).cpu()

    os.makedirs(OUT_DIR, exist_ok=True)

    png_path = os.path.join(OUT_DIR, f"{output_name}.png")
    latent_path = os.path.join(OUT_DIR, f"{output_name}_latents.pt")

    # Zapis siatki bez podpisów
    save_image(
        fake,
        png_path,
        nrow=10,
        normalize=True,
        value_range=(-1, 1),
    )

    # Zapis latentów do reprodukcji
    torch.save(
        {
            "z1": z1.cpu(),
            "z2": z2.cpu(),
            "z_all": z_all.cpu(),
            "alphas": alphas.cpu(),
            "checkpoint_path": checkpoint_path,
            "interpolation_seed": interpolation_seed,
            "z_dim": z_dim,
            "epoch": ckpt.get("epoch"),
            "fid": ckpt.get("fid"),
            "lr": ckpt.get("lr"),
            "batch_size": ckpt.get("batch_size"),
            "seed": ckpt.get("seed"),
        },
        latent_path,
    )

    # Podgląd z podpisami alpha
    fig, axes = plt.subplots(1, 10, figsize=(20, 3))

    for i, ax in enumerate(axes):
        img = to_01(fake[i]).permute(1, 2, 0)
        ax.imshow(img)
        ax.set_title(f"α={alphas[i].item():.2f}", fontsize=9)
        ax.axis("off")

    epoch = ckpt.get("epoch", "?")
    fid = ckpt.get("fid", None)

    if fid is not None:
        fig.suptitle(f"DCGAN latent interpolation | epoch={epoch} | FID={fid:.2f}", fontsize=13)
    else:
        fig.suptitle(f"DCGAN latent interpolation | epoch={epoch}", fontsize=13)

    plt.tight_layout()

    preview_path = os.path.join(OUT_DIR, f"{output_name}_preview.png")
    plt.savefig(preview_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved image grid to:", png_path)
    print("Saved preview image to:", preview_path)
    print("Saved latent vectors to:", latent_path)

    return fake, z_all.cpu(), alphas.cpu()

In [10]:
@torch.no_grad()
def make_dcgan_latent_interpolation_50_images(
    checkpoint_path,
    output_dir_name="interpolation_50_pairs",
    interpolation_seed=123,
    n_pairs=50,
):
    G, z_dim, ckpt = load_generator_for_interpolation(checkpoint_path)

    z_gen = torch.Generator(device=device)
    z_gen.manual_seed(interpolation_seed)

    alphas = torch.linspace(0, 1, steps=10, device=device)

    save_dir = os.path.join(OUT_DIR, output_dir_name)
    os.makedirs(save_dir, exist_ok=True)

    all_metadata = []

    for pair_idx in range(n_pairs):
        z1 = torch.randn(1, z_dim, 1, 1, device=device, generator=z_gen)
        z2 = torch.randn(1, z_dim, 1, 1, device=device, generator=z_gen)

        z_all = torch.cat(
            [(1 - alpha) * z1 + alpha * z2 for alpha in alphas],
            dim=0
        )  # [10, z_dim, 1, 1]

        fake = G(z_all).cpu()  # [10, C, H, W]

        img_path = os.path.join(save_dir, f"pair_{pair_idx+1:02d}.png")
        latents_path = os.path.join(save_dir, f"pair_{pair_idx+1:02d}_latents.pt")

        # zapis jednego stripu: 10 obrazów w jednym rzędzie
        save_image(
            fake,
            img_path,
            nrow=10,
            normalize=True,
            value_range=(-1, 1),
        )

        # zapis latentów dla tej jednej pary
        torch.save(
            {
                "pair_index": pair_idx + 1,
                "z1": z1.cpu(),
                "z2": z2.cpu(),
                "z_all": z_all.cpu(),
                "alphas": alphas.cpu(),
                "checkpoint_path": checkpoint_path,
                "interpolation_seed": interpolation_seed,
                "z_dim": z_dim,
                "epoch": ckpt.get("epoch"),
                "fid": ckpt.get("fid"),
                "lr": ckpt.get("lr"),
                "batch_size": ckpt.get("batch_size"),
                "seed": ckpt.get("seed"),
            },
            latents_path,
        )

        all_metadata.append({
            "pair_index": pair_idx + 1,
            "image_path": img_path,
            "latents_path": latents_path,
        })

    print(f"Saved {n_pairs} separate interpolation images to: {save_dir}")
    return all_metadata

In [ ]:
run_name = "lr=0.001_z=64_bs=32_seed=142"

checkpoint_path = os.path.join(OUT_DIR, f"best_{run_name}.pt")

results = make_dcgan_latent_interpolation_50_images(
    checkpoint_path=checkpoint_path,
    output_dir_name=f"interpolation_50_pairs_best_{run_name}",
    interpolation_seed=123,
    n_pairs=50,
)

Saved 50 separate interpolation images to: .\interpolation_50_pairs_best_lr=0.001_z=64_bs=32_seed=142


: 